# RoPE band ablation — long-context extension (8k / 16k)

Tests the finding from the 1k–4k run: *a band starts to matter once its wavelength is comparable to the context length* (band 4, λ 6k–28k, became important for vt/fwe only at 4096). At 8k/16k it should strengthen, and band 5 (λ 35k–160k) may start to matter.

Same model, same factor (32, from calibration), same seeds. Tasks: s_niah, vt, fwe (mk_niah dropped — it tracked s_niah exactly; spec §7 cut order). Results from the 1k–4k run come with the repo and are skipped.

**8192 is done** (all of s_niah/vt/fwe, n=50, committed to the repo). For **16384** a full run would take ~19 h on T4 (≈45 s/example), so the default here is a reduced design aimed at the open questions: tasks vt (main) + s_niah (control), bands 3–7 (the ones whose importance changes with length; bands 0–2 are already settled), n=30. A kernel-consistency check and a timing probe run first and lowers n to fit the time budget, or stops if even n=20 does not fit.

**Run one length per commit** (≈4 h for 8192, ≈8–9 h for 16384 on T4): set `LONG_LENGTH`, then *Save Version → Save & Run All*. Settings: GPU T4 x2, Internet On, secret `GITHUB_TOKEN`. To keep the 8192 results when running 16384, attach the 8192 version's output as input (cell 2 restores it) — or send the zip to be committed to the repo.

In [ ]:
# 1. Setup — edit these
LONG_LENGTH = 16384       # 8192 already done and committed
TASKS = ["vt", "s_niah"]  # vt = main signal, s_niah = control
BANDS = [3, 4, 5, 6, 7]   # bands whose importance moves with length (0-2 settled at 1k-8k)
N = 50                    # examples per cell; lowered automatically to fit TIME_BUDGET_H
N_MIN = 20
TIME_BUDGET_H = 10.0      # Kaggle commit limit is 12 h
MIN_BASELINE = 0.2        # a task with lower baseline at this length carries no signal -> skipped in the sweep
REPO_URL = "https://github.com/leeduy0403/rope-freq-ablation.git"
PRIVATE_REPO = True
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

import os, subprocess, glob, json, shutil

def sh(cmd):
    """Run a shell command, stream its output, raise if it fails (stops Run All)."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {cmd}")

url = REPO_URL
if PRIVATE_REPO:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
REPO = "/kaggle/working/rope-freq-ablation"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q", url, "main"], check=True)
else:
    subprocess.run(["git", "clone", "-q", url, REPO], check=True)
os.chdir(REPO)
sh("git log --oneline -1 && pip install -q pyyaml")
assert "configure_attention" in open("src/model_io.py").read(), "repo is older than the 16k fix — re-run this cell / check REPO_URL"
M = f"--model-id {MODEL_ID}"
slug = MODEL_ID.replace("/", "__")


In [ ]:
# 2. Restore results from a previous version (if attached as input)
for src in glob.glob("/kaggle/input/*/rope-freq-ablation/results") + glob.glob("/kaggle/input/*/results"):
    print("restoring from", src)
    shutil.copytree(src, "results", dirs_exist_ok=True)
print("existing sweep cells:", len(glob.glob(f"results/{slug}/sweep/compress_f32/*.json")))


In [ ]:
# Environment + unit tests
sh("python -m src.env_report")
sh("python -m pytest -q tests")


In [ ]:
# Kernel consistency check.
# Runs up to 8192 used SDPA's math kernel (silent fallback, see src/model_io.configure_attention); from now on the
# memory-efficient kernel is forced (the math kernel OOMs at 16k). Re-run the 4096 baseline with the new kernel and
# compare per-example correctness with the stored results, so 16k cells can be compared with the 1k-8k ones.
CHECK_LEN = 4096
sh(f"python -m src.eval.runner --stage baseline {M} --tasks {' '.join(TASKS)} --lengths {CHECK_LEN} "
   f"--results-dir results/_kernel_check --no-gate")
agree = {}
for t in TASKS:
    old = json.load(open(f"results/{slug}/baseline/{t}_{CHECK_LEN}.json"))
    new = json.load(open(f"results/_kernel_check/{slug}/baseline/{t}_{CHECK_LEN}.json"))
    a = [x["correct"] for x in old["examples"]]
    b = [x["correct"] for x in new["examples"]]
    agree[t] = sum(x == y for x, y in zip(a, b)) / len(a)
    print(f"{t}@{CHECK_LEN}: math acc={old['accuracy']:.2f}  efficient acc={new['accuracy']:.2f}  "
          f"per-example agreement={agree[t]:.2f}  ({new['attention']})")
json.dump(agree, open(f"results/{slug}/kernel_check_{CHECK_LEN}.json", "w"), indent=2)
if min(agree.values()) < 0.9:
    raise RuntimeError("kernels disagree on >10% of examples — 16k cells would not be comparable; stop and report")


In [ ]:
# Timing / memory probe: 2 examples of vt at LONG_LENGTH (also catches OOM before the long run)
sh(f"python -m src.eval.runner --stage baseline {M} --tasks vt --lengths {LONG_LENGTH} --n 2 "
   f"--results-dir results/_probe --no-gate --force")
sec = json.load(open(f"results/_probe/{slug}/baseline/vt_{LONG_LENGTH}.json"))["wall_seconds"] / 2
n_gen_per_example = len(TASKS) * (len(BANDS) + 2)        # baseline + band=none + bands
fit = int(TIME_BUDGET_H * 3600 / (sec * 1.1 * n_gen_per_example))
print(f"{sec:.1f} s/example (vt) -> n={N} needs {sec * 1.1 * n_gen_per_example * N / 3600:.1f} h; "
      f"n that fits {TIME_BUDGET_H} h: {fit}")
N = min(N, fit)
if N < N_MIN:
    raise RuntimeError(f"only n={N} fits the time budget — cut TASKS or BANDS and re-run")
print("using n =", N)


In [ ]:
# Baseline at LONG_LENGTH (G2 thresholds are defined for the shortest length, so not enforced here)
sh(f"python -m src.eval.runner --stage baseline {M} --tasks {' '.join(TASKS)} --lengths {LONG_LENGTH} --n {N} --no-gate")
base = {t: json.load(open(f"results/{slug}/baseline/{t}_{LONG_LENGTH}.json"))["accuracy"] for t in TASKS}
SWEEP_TASKS = [t for t in TASKS if base[t] >= MIN_BASELINE]
for t in TASKS:
    print(f"{t}@{LONG_LENGTH}: baseline {base[t]:.2f}", "" if t in SWEEP_TASKS else "-> SKIPPED (no signal)")
if not SWEEP_TASKS:
    raise RuntimeError("no task has signal at this length — stop and report")


In [ ]:
# Sweep at LONG_LENGTH, factor from config (32). Gate G4 enforced (band=none must reproduce baseline).
sh(f"python -m src.eval.runner --stage sweep {M} --tasks {' '.join(SWEEP_TASKS)} --lengths {LONG_LENGTH} --n {N} --bands {' '.join(map(str, BANDS))}")


In [ ]:
# Analysis over every length available (1k-4k from the repo + this run)
sh(f"python -m src.analyze.heatmap {M}")
sh(f"python -m src.analyze.overlay {M} --scale 4")
sh(f"python -m src.analyze.shift {M}")
from IPython.display import Image, display
for p in sorted(glob.glob("figures/*.png")):
    print(p); display(Image(p))


In [ ]:
# Package results for download
sh("cd /kaggle/working && zip -qr results.zip rope-freq-ablation/results rope-freq-ablation/figures && ls -la results.zip")
